<a href="https://colab.research.google.com/github/TienNguyen0712/hybrid-llm-tabular-pipeline-for-icu-mortality-prediction/blob/main/mimic_iv_rag_pipeline_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Trích xuất thông tin & So sánh hồ sơ tương đòng từ bộ MIMIC-IV dể giải quyết bài toán Dự đoán mức độ sinh tồn**

In [1]:
# ── CELL 1: Cài thư viện + Mount Drive ──────────────────────
# Bỏ comment nếu chạy trên Google Colab
from google.colab import drive
drive.mount('/content/drive')
!pip install tableone pyarrow seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import gc
import warnings
warnings.filterwarnings('ignore')

# Cài đặt style biểu đồ chuyên nghiệp
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")
print("✓ Libraries loaded")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
✓ Libraries loaded


## **Cấu hình đường dẫn**

In [2]:
# ── CELL 2: Đường dẫn dữ liệu ───────────────────────────────
# === CHỈNH Ở ĐÂY ===
# DATA_DIR = r"E:\KLTN\mimiciv\3.1\parquet"  # Windows local
DATA_DIR = "/content/drive/MyDrive/NCKH-DDU1231/mimic-iv-clinical-database-demo-2.2/mimic-iv-clinical-database-demo-2.2"  # Google Colab

import os
def load(folder, name):
    path = os.path.join(DATA_DIR, folder, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✓ {folder}/{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
        return df
    else:
        print(f"  ✗ {folder}/{name}: KHÔNG TÌM THẤY")
        return pd.DataFrame()

## **Nạp dữ liệu cơ bản**

In [17]:
# ── CELL 3: Thống kê mô tả ───────────────────────────────

def describe_col(name):
  for col in name.columns:
    print(f"{col}: {patients[col].nunique()} unique values - {patients[col].dtype} dtype")

### **Xây dựng bảng cohort dựa theo các điều kiện**

Xây dựng một bảng cohort được gộp từ bảng patients, admissions, icustay. Điều kiện gộp và lọc bao gồm:
- Các bệnh nhân phải là người trưởng thành (trên 18 tuổi)
- Lấy đợt năm ICU dầu tiên và loại bỏ các đợt nằm ICU sau
- Thời gian nằm viện phải trên 24h kể từ khi vào ICU



In [4]:
# ── CELL 3: xây dựng cohort ───────────────────────────────
print("Loading core tables...")

# hosp
patients   = load("hosp", "patients.csv.gz")      # Thông tin bệnh nhân
admissions = load("hosp", "admissions.csv.gz")    # Thông tin nhập viện

# icu
icustays   = load("icu",  "icustays.csv.gz")      # Thông tin lần nằm ICU

# Merge thành cohort
cohort = (
    icustays[["subject_id", "hadm_id", "stay_id", "intime", "first_careunit", "outtime"]]
    .merge(admissions[["subject_id", "hadm_id", "admittime", "dischtime", "admission_type",
                       "admission_location", "edregtime", "edouttime", "deathtime",
                       "hospital_expire_flag", "insurance", "race"]],
           on=["subject_id", "hadm_id"], how="left")
    .merge(patients[["subject_id", "gender", "anchor_age", "anchor_year", "dod"]],
           on="subject_id", how="left")
)

# Chỉ người lớn, first ICU stay per admission
cohort = cohort[cohort["anchor_age"] >= 18]
cohort = (cohort.sort_values(["subject_id", "hadm_id", "intime"])
                .groupby(["subject_id", "hadm_id"], as_index=False).first())

# Tính các biến dẫn xuất
cohort["icu_los_hours"] = (
    pd.to_datetime(cohort["outtime"]) - pd.to_datetime(cohort["intime"])
).dt.total_seconds() / 3600                                                       # Thời gian nằm ICU theo giờ
cohort["hosp_los_days"] = (
    pd.to_datetime(cohort["dischtime"]) - pd.to_datetime(cohort["admittime"])
).dt.total_seconds() / 86400                                                      # Thời gian nằm viện theo ngày
cohort["mortality"] = cohort["hospital_expire_flag"].astype(int)                  # Nhãn mục tiêu
cohort["age_group"] = pd.cut(cohort["anchor_age"],
                              bins=[17, 30, 50, 65, 80, 120],
                              labels=["18-30", "31-50", "51-65", "66-80", "80+"]) # Phân loại độ tuổi thành 5 nhóm
print(f"\nCohort: {len(cohort):,} ICU stays | "
      f"{cohort['subject_id'].nunique():,} unique patients")
print(f"Mortality: {cohort['mortality'].sum():,} ({cohort['mortality'].mean()*100:.1f}%)")

Loading core tables...
  ✓ hosp/patients.csv.gz: 100 rows × 6 cols
  ✓ hosp/admissions.csv.gz: 275 rows × 16 cols
  ✓ icu/icustays.csv.gz: 140 rows × 8 cols

Cohort: 128 ICU stays | 100 unique patients
Mortality: 15 (11.7%)


In [5]:
cohort.head()

,subject_id,hadm_id,stay_id,intime,first_careunit,outtime,admittime,dischtime,admission_type,admission_location,...,insurance,race,gender,anchor_age,anchor_year,dod,icu_los_hours,hosp_los_days,mortality,age_group
0,10000032,29079034,39553978,2180-07-23 14:00:00,Medical Intensive Care Unit (MICU),2180-07-23 23:50:47,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.,EMERGENCY ROOM,...,Medicaid,WHITE,F,52,2180,2180-09-09,9.846389,2.222222,0,51-65
1,10001217,24597018,37067082,2157-11-20 19:18:02,Surgical Intensive Care Unit (SICU),2157-11-21 22:08:00,2157-11-18 22:56:00,2157-11-25 18:00:00,EW EMER.,EMERGENCY ROOM,...,Other,WHITE,F,55,2157,None,26.832778,6.794444,0,51-65
2,10001217,27703517,34592300,2157-12-19 15:42:24,Surgical Intensive Care Unit (SICU),2157-12-20 14:27:41,2157-12-18 16:58:00,2157-12-24 14:55:00,DIRECT EMER.,PHYSICIAN REFERRAL,...,Other,WHITE,F,55,2157,None,22.754722,5.914583,0,51-65
3,10001725,25563031,31205490,2110-04-11 15:52:22,Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-12 23:59:56,2110-04-11 15:08:00,2110-04-14 15:00:00,EW EMER.,PACU,...,Other,WHITE,F,46,2110,None,32.126111,2.994444,0,31-50
4,10002428,20321825,34807493,2156-04-30 21:53:00,Medical Intensive Care Unit (MICU),2156-05-02 22:27:20,2156-04-30 20:35:00,2156-05-03 16:36:00,EW EMER.,EMERGENCY ROOM,...,Medicare,WHITE,F,80,2155,None,48.572222,2.834028,0,66-80


### **Xây dựng bảng chart vitals từ bảng `chartevents`**


In [21]:
# hosp
labevents = load("hosp", "labevents.csv.gz")
prescriptions   = load("hosp",  "prescriptions.csv.gz")

# icu
chartevents   = load("icu",  "chartevents.csv.gz")
inputevents    = load("icu",  "inputevents.csv.gz")
outputevents   = load("icu",  "outputevents.csv.gz")

  ✓ hosp/labevents.csv.gz: 107,727 rows × 16 cols
  ✓ hosp/prescriptions.csv.gz: 18,087 rows × 21 cols
  ✓ icu/chartevents.csv.gz: 668,862 rows × 11 cols
  ✓ icu/inputevents.csv.gz: 20,404 rows × 26 cols
  ✓ icu/outputevents.csv.gz: 9,362 rows × 9 cols


In [22]:
select_table = [labevents, prescriptions, inputevents, outputevents]